Import libraries and functions

In [1]:
import numpy as np
from torch import nn
from poisson_bcann_vi import *
from codebase_0 import CANN, MLP, cann_training_loop, mlp_train_loop, DeviceLoader
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler

from poisson_bcann_hmc_nuts import (
    stratified_subsample, make_chain_inits, run_nuts, time_nuts, get_device, to_device
)
from poisson_bcann_vi import PoissonBCANN, build_bcann_guide
from bcann_neutra import neutra_fit_guide, time_neutra, run_neutra_parallel
from poisson_horseshoe import HorseshoePoissonBCANN


seed = 42
set_seed(seed)
pyro.clear_param_store()

/workspace/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Load data and perfrom train/validation/test split

In [2]:
device = torch.device("cuda:1")

torch.set_default_device("cpu")

dataset = pd.read_csv("freMTPL2freq_transformed.csv", index_col= 0) 
y = torch.tensor(dataset["ClaimNb"].to_numpy(), dtype=torch.float32)
exposure = torch.tensor(dataset["Exposure"].to_numpy(), dtype=torch.float32)
X = torch.tensor(dataset.drop(columns=["IDpol", "Exposure","ClaimNb"]).to_numpy(), dtype=torch.float32)   # 48 columns

# Learning-test set split
X_learn, X_test, y_learn, y_test, exp_learn, exp_test = train_test_split(
    X, y, exposure, test_size= 0.1, random_state= seed
)

# Training-CV set split
X_train, X_val, y_train, y_val, exp_train, exp_val = train_test_split(
    X_learn, y_learn, exp_learn, test_size= 0.1, random_state= seed
)

learn_scaler = MinMaxScaler().fit(X_learn)
X_learn = learn_scaler.transform(X_learn)
X_test = learn_scaler.transform(X_test)

train_scaler = MinMaxScaler().fit(X_train)
X_train = train_scaler.transform(X_train)
X_val = train_scaler.transform(X_val)

def T(a):
    return torch.as_tensor(a, dtype=torch.float32, device=device)

X_train, X_val, X_learn, X_test = map(T, (X_train, X_val, X_learn, X_test))
y_train, y_val, y_learn, y_test = map(T, (y_train, y_val, y_learn, y_test))
exp_train, exp_val, exp_learn, exp_test = map(T, (exp_train, exp_val, exp_learn, exp_test))

torch.set_default_device(device)

train_loader = DeviceLoader(
        X_train, y_train, torch.log(exp_train),
        batch_size = 10000,
        shuffle= True
)

val_loader = DeviceLoader(
        X_val, y_val, torch.log(exp_val),
        batch_size= 10000
)

learn_loader = DeviceLoader(
        X_learn, y_learn, torch.log(exp_learn),
        batch_size= 10000
)

Subsample for Neutra run

In [3]:
sub_ids = stratified_subsample(y_learn, 30000, seed = seed)
X_sub, y_sub, logexp_sub = X_learn[sub_ids], y_learn[sub_ids], torch.log(exp_learn[sub_ids])

Start HorseshoePoissonBCANN, and train the IAF with guide function

In [4]:
network_architecture = [20,15, 10]

# flow (step 2)
NUM_TRANSFORMS = 2        # IAF layers; 0 -> diagonal Gaussian (NeuTra only rescales the space)
FLOW_HIDDEN = None      # IAF hidden widths; None -> [latent_dim]. Try [512] if the flow is slow
SVI_BATCH = None          # None -> full-batch ELBO on the subsample
SVI_STEPS = 5000
SVI_LR, SVI_LR_FINAL = 5e-3, 5e-4

glm_mle = poisson_fit_glm_mle(X_sub, y_sub, logexp_sub, random_state=seed)

with torch.device(device):
        poisson_horseshoe_bcann = HorseshoePoissonBCANN(
                in_features= 48,
                neural_net_params={"hidden_activation_nodes": network_architecture,
                                    "hidden_activations": nn.Tanh},
            )
        horseshoe_poisson_guide, horseshoe_poisson_history = neutra_fit_guide(
            poisson_horseshoe_bcann, 
            X_sub, 
            y_sub, 
            logexp_sub, 
            glm_mle=tuple(t.to(device) for t in glm_mle),
            num_transforms=NUM_TRANSFORMS, hidden_dim= FLOW_HIDDEN,
            num_steps=SVI_STEPS, lr=SVI_LR, lr_final=SVI_LR_FINAL, batch_size=10000,
            report_every=max(SVI_STEPS // 20, 1), random_state=seed
        )


step 250 | -ELBO/n: 0.3158 | 15s
step 500 | -ELBO/n: 0.2620 | 27s
step 750 | -ELBO/n: 0.2381 | 40s
step 1000 | -ELBO/n: 0.2317 | 52s
step 1250 | -ELBO/n: 0.2282 | 65s
step 1500 | -ELBO/n: 0.2269 | 77s
step 1750 | -ELBO/n: 0.2234 | 90s
step 2000 | -ELBO/n: 0.2222 | 102s
step 2250 | -ELBO/n: 0.2200 | 115s
step 2500 | -ELBO/n: 0.2198 | 127s
step 2750 | -ELBO/n: 0.2178 | 140s
step 3000 | -ELBO/n: 0.2183 | 152s
step 3250 | -ELBO/n: 0.2193 | 164s
step 3500 | -ELBO/n: 0.2161 | 177s
step 3750 | -ELBO/n: 0.2158 | 189s
step 4000 | -ELBO/n: 0.2163 | 201s
step 4250 | -ELBO/n: 0.2163 | 214s
step 4500 | -ELBO/n: 0.2162 | 226s
step 4750 | -ELBO/n: 0.2151 | 239s
step 5000 | -ELBO/n: 0.2160 | 251s


Neutra hyperparameters

In [8]:
# NUTS (step 3)
CHAINS = 6
WARMUP, SAMPLES = 1500, 1500
TARGET_ACCEPT = 0.9
MAX_TREE_DEPTH = 8

nuts_kwargs = dict(num_warmup=WARMUP, num_samples=SAMPLES, target_accept=TARGET_ACCEPT,
                       max_tree_depth=MAX_TREE_DEPTH, init_scale=1.0, jit_compile = True)

Test Neutra run

In [ ]:
time_neutra(
    poisson_horseshoe_bcann, 
    horseshoe_poisson_guide, 
    X_sub, 
    y_sub, 
    logexp_sub, 
    n_iter=20, device=device
)  # cost estimate first

3.99 s / iteration on cuda:1  (early warmup iterations are not representative of tree depth)


3.9934717416763306

Neutra Run with full config

In [9]:
samples, info = run_neutra_parallel(
    HorseshoePoissonBCANN, 
    dict(
        in_features=48, 
        neural_net_params={
            "hidden_activation_nodes": network_architecture,
            "hidden_activations": nn.Tanh
        }
    ), 
    horseshoe_poisson_guide, 
    X_sub, 
    y_sub,
    logexp_sub,
    num_chains=CHAINS, devices=device,
    out_dir=f"neutra_chains_poisson_horseshoe", seed=seed,
    **nuts_kwargs
)

elapsed 3:53:26
chain 5: Sample: 100%|██████████| 3000/3000 [3:46:15,  4.53s/it, step size=1.03e-03, acc. prob=0.878]


Write to disk for later uses

In [12]:
torch.save(
    {
        "samples": samples,
        "info": info,
        "config": dict(num_warmup=WARMUP, num_samples=SAMPLES, target_accept=TARGET_ACCEPT,
                       max_tree_depth=MAX_TREE_DEPTH, seed=seed, n_chains=6),
    },
    "hmc_results/poisson_horseshoe_neutra_full_set.pt",
)
